# Feature Engineering Pipeline
Read raw data, compute features, write back to MinIO.

In [ ]:
import boto3
import os
import io
import pandas as pd
import numpy as np
from botocore.client import Config

s3 = boto3.client(
    's3',
    endpoint_url=os.getenv('MINIO_ENDPOINT', 'http://localhost:9000'),
    aws_access_key_id=os.getenv('MINIO_ACCESS_KEY', 'minioadmin'),
    aws_secret_access_key=os.getenv('MINIO_SECRET_KEY', 'minioadmin'),
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

bucket = 'raw-data'
existing = [b['Name'] for b in s3.list_buckets()['Buckets']]

df = None
if bucket in existing:
    objs = s3.list_objects_v2(Bucket=bucket).get('Contents', [])
    keys = [o['Key'] for o in objs if o['Key'].endswith('.parquet')]
    if keys:
        obj = s3.get_object(Bucket=bucket, Key=keys[0])
        df = pd.read_parquet(io.BytesIO(obj['Body'].read()))
        print(f'Loaded {len(df)} rows from s3://{bucket}/{keys[0]}')

if df is None:
    print('No raw data found — generating synthetic ecommerce events dataset')
    np.random.seed(0)
    n = 10000
    df = pd.DataFrame({
        'event_ts': pd.date_range('2024-01-01', periods=n, freq='5min'),
        'user_id': np.random.randint(1, 500, n),
        'product_id': np.random.randint(1, 200, n),
        'category': np.random.choice(['electronics', 'clothing', 'food', 'sports'], n),
        'amount': np.random.exponential(60, n),
        'quantity': np.random.randint(1, 10, n)
    })
    print(df.head())

In [ ]:
if 'event_ts' not in df.columns:
    df['event_ts'] = pd.date_range('2024-01-01', periods=len(df), freq='5min')

df = df.sort_values('event_ts').reset_index(drop=True)

# Rolling 7-day window of amount per user (approximated by row window here)
df['amount_roll_7'] = (
    df.groupby('user_id')['amount']
    .transform(lambda x: x.rolling(window=7, min_periods=1).mean())
)
df['quantity_roll_7'] = (
    df.groupby('user_id')['quantity']
    .transform(lambda x: x.rolling(window=7, min_periods=1).sum())
)
print('Rolling averages computed')
print(df[['user_id', 'amount', 'amount_roll_7', 'quantity_roll_7']].head(10))

In [ ]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
if cat_cols:
    df = pd.get_dummies(df, columns=cat_cols, drop_first=False, dtype=int)
    print(f'One-hot encoded columns: {cat_cols}')
    print(f'DataFrame shape after encoding: {df.shape}')
else:
    print('No categorical columns to encode')

In [ ]:
ts_col = 'event_ts'
if ts_col in df.columns:
    df['hour'] = df[ts_col].dt.hour
    df['day_of_week'] = df[ts_col].dt.dayofweek
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
    df['month'] = df[ts_col].dt.month
    df = df.drop(columns=[ts_col])
    print('Time-based features added: hour, day_of_week, is_weekend, month')
else:
    print('No timestamp column found, skipping time features')

print(df.head())

In [ ]:
features_bucket = 'features'
out_key = 'engineered/features_v1.parquet'

existing = [b['Name'] for b in s3.list_buckets()['Buckets']]
if features_bucket not in existing:
    s3.create_bucket(Bucket=features_bucket)
    print(f'Created bucket: {features_bucket}')

buf = io.BytesIO()
df.to_parquet(buf, index=False)
buf.seek(0)
s3.put_object(Bucket=features_bucket, Key=out_key, Body=buf.read())
print(f'Feature dataset written to s3://{features_bucket}/{out_key}')
print(f'Shape: {df.shape}')